# GEM TREQ: grilla de microzonas de Cali

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad

Una fila es un punto de la grilla de sitio de GEM, no una medición en un predio.

In [ ]:
PATH=ROOT/"data/externos/gem_microzonas_grilla.csv"
meta=ficha(PATH)
# Lectura textual para auditar nulos y tipos sin perder códigos.
df=pd.read_csv(PATH,dtype=str,keep_default_na=False)
calidad=mostrar_perfil(df)
x=df.copy()
for c in ["lon","lat","vs30","z1pt0","z2pt5","vs30measured"]:x[c]=pd.to_numeric(x[c],errors="coerce")
g=gpd.GeoDataFrame(x,geometry=gpd.points_from_xy(x.lon,x.lat),crs=4326)
geo=geometria(g)
display(pd.Series(meta).to_frame("procedencia"));display(pd.Series(geo).to_frame("geometria"))
resumen={"dataset":"10_gem_microzonas",**meta,"filas":len(df),"calidad_columnas":calidad.to_dict("records"),"geometria":geo}

## 2. Códigos, valores de referencia y resolución de la grilla

In [ ]:
codigos=conteos(df.ampcode,"ampcode")
display(codigos)
display(x.groupby("ampcode").agg(puntos=("vs30","size"),vs30_min=("vs30","min"),vs30_max=("vs30","max")))
display(conteos(df.vs30measured,"vs30measured"))
from scipy.spatial import cKDTree
xy=np.column_stack([g.to_crs(CRS_METROS).geometry.x,g.to_crs(CRS_METROS).geometry.y])
dist=cKDTree(xy).query(xy,k=2)[0][:,1]
display(pd.Series(dist).describe().to_frame("vecino_mas_cercano_m"))
fig,ax=plt.subplots(figsize=(9,6));g.plot(ax=ax,column="ampcode",categorical=True,legend=True,markersize=9)
ax.set(title="Grilla GEM: códigos de amplificación",xlabel="Longitud",ylabel="Latitud")
fig.tight_layout();fig.savefig(SALIDAS/"10_grilla.png",dpi=150);plt.show()
resumen.update({"codigos":df.ampcode.value_counts().to_dict(),"duplicados_lon_lat":int(df.duplicated(["lon","lat"]).sum()),
 "vs30_por_codigo":x.groupby("ampcode").vs30.agg(["min","max"]).reset_index().to_dict("records"),
 "espaciado_vecino_m":pd.Series(dist).describe().to_dict()})

## Reglas decididas para plata

- **Incorporar como contexto de sitio**, unido a las funciones de amplificación mediante `ampcode`.
- Preservar coordenadas, código, corte y referencia del modelo. La Vs30=800 dentro de microzonas actúa como roca de referencia en este esquema; no concluir que todos esos suelos son roca medida.
- Guardar distancia al punto asignado y bandera de soporte espacial. Un vecino lejano o código ausente requiere revisión; no convertir faltantes en `noaf`.
- `noaf` es una categoría explícita del modelo, distinta de dato perdido. El significado de `vs30measured` debe conservarse según la documentación de la fuente, sin inferir una campaña de medición por predio.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('10_gem_microzonas', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")